In [ ]:
#Table
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
from snowflake.snowpark.functions import col, dateadd, lit, current_date, when, countDistinct
import pandas as pd
import datetime as dt
import os 
from dotenv import load_dotenv

load_dotenv()

connection_parameters = {
    "account": os.getenv(SNOWFLAKE_ACCOUNT),
    "user": os.getenv(SNOWFLAKE_USER),
    "authenticator": os.getenv(SNOWFLAKE_AUTHENTICATOR),
    "insecure_mode":True,
}

session = Session.builder.configs(connection_parameters).create()

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://dish.okta.com/app/snowflake/exk9ewak82R5btDoW2p7/sso/saml?SAMLRequest=pZJPc9owEMW%2Fikc925JNaLAGkyHx0DCTNIwxhPYmbAEa25KjlTHh01fmTyc9JJfeNNJb%2Fd7u2%2BHdoSqdPdcglIyQ7xHkcJmpXMhthBbpxB0gBwyTOSuV5BF654DuRkNgVVnTcWN2MuFvDQfj2I8k0O4hQo2WVDEQQCWrOFCT0fn4%2BYkGHqEMgGtjcehSkoOwrJ0xNcW4bVuv7XlKb3FACMEkxFbVSb6hD4j6a0atlVGZKq8lB9vTJwgfk5sOYRWWMLsU3gt5HsFXlPVZBPQxTWfu7GWeImd87e5BSWgqrudc70XGF8nT2QBYB7%2BW44D4vYHXgNva2bmBV2uxZ4aXQhYeSNVuSlbwTFV1YyzCsye84Tku1VbYwU3jCNWFyN9Wr228mpBgPX883h5XZvlj%2F0x4Et%2BHL9%2FnKRFBVTVLduTJIkPO8hpz0MU8BWj4VHbhGntFgr5LQpcMUj%2BkNwHt9T2%2FF%2F5GTmwNCsnMqfLaQS5g56nCsJMzVtf4r2nMD0XIW1YMgqS%2FNrF6DepbDKBwFxw67w490fXovyYyxB%2B%2FuuzkTxvTNJ6pUmTvzkTpipnPU%2FQ9%2F3QjcndzklJeMVGO81xzAJtmWar2QXPrI0JGNxzh0Zn67%2FKP%2FgA%3D&Re

In [ ]:
t = session.table(os.getenv("ORDER_SCHEMA") + "." + os.getenv("ORDER_TABLE")).limit(100).to_pandas()


In [ ]:
snowflake_orders = session.table(os.getenv("ORDER_SCHEMA") + "." + os.getenv("ORDER_TABLE"))
status_crosstab_df = snowflake_orders.crosstab("ORDR_STATUS", "ORDR_STATUSDETAIL")
status_crosstab_df.show()

In [8]:
unique_status = status_crosstab_df.select(col('ORDR_STATUS')).distinct().collect()
print([row['ORDR_STATUS'] for row in unique_status])

['returnAccepted', 'inProgress', 'hold', 'validated', 'failed', 'wait', 'cancelled', 'Cancelled', 'canceled', 'Removed', 'cancel', 'completed', 'complete', 'error', 'Complete', 'removed', 'activated']


In [12]:
multi_status_check_df = (snowflake_orders
    .groupBy("ORDER_ITEM_ID")
    .agg(countDistinct("ORDR_STATUS").alias("DISTINCT_STATUS_COUNT"))
    .filter(col("DISTINCT_STATUS_COUNT") > 1)
)


print("Order items with more than one status (if any):")
multi_status_check_df.show()


Order items with more than one status (if any):
-------------------------------------------------
|"ORDER_ITEM_ID"      |"DISTINCT_STATUS_COUNT"  |
-------------------------------------------------
|9673-254359-8858-01  |2                        |
|1417-158707-7630-01  |2                        |
|4001-598112-1671-04  |2                        |
|5235-024148-9348-01  |2                        |
|6150-833538-7809-01  |2                        |
|4750-045999-4793-01  |2                        |
|4143-881088-0162-01  |2                        |
|7228-428099-0480-05  |2                        |
|5616-754644-9039-04  |2                        |
|3811-407751-8180-01  |2                        |
-------------------------------------------------



In [16]:
print("\nDisplaying all statuses for each item with multiple status entries:")

# Use the list of items from the previous step to filter the original DataFrame
# and show the specific statuses for each. We sort the results for readability.
detailed_statuses_df = (snowflake_orders
    .join(multi_status_check_df, "ORDER_ITEM_ID")
    .select(snowflake_orders["ORDER_ITEM_ID"], snowflake_orders["ORDR_STATUS"], snowflake_orders["ORDR_STATUSDETAIL"], snowflake_orders['OO_LASTMODIFIED_DT_MT'])
    .sort(snowflake_orders["ORDER_ITEM_ID"], snowflake_orders["ORDR_STATUS"])
)

# Show the detailed view. This will be empty if multi_status_check_df was empty.
detailed_statuses_df.show()


Displaying all statuses for each item with multiple status entries:
------------------------------------------------------------------------------------------
|"ORDER_ITEM_ID"      |"ORDR_STATUS"  |"ORDR_STATUSDETAIL"  |"OO_LASTMODIFIED_DT_MT"     |
------------------------------------------------------------------------------------------
|0001-015573-8430-01  |complete       |complete             |2025-08-21 00:40:30.552000  |
|0001-015573-8430-01  |hold           |pendingReturn        |2025-08-21 00:59:43.899000  |
|0003-729173-0292-01  |complete       |complete             |2025-08-20 18:54:42.259000  |
|0003-729173-0292-01  |inProgress     |provisioned          |2025-08-21 01:07:34.653000  |
|0003-729173-0292-05  |complete       |complete             |2025-08-21 05:42:26.998000  |
|0003-729173-0292-05  |inProgress     |provisioned          |2025-08-21 01:07:34.653000  |
|0004-812707-6920-01  |cancelled      |cancelled            |2025-08-29 10:39:22.894000  |
|0004-812707-6920-01 